# Day 071 — Exercise 2: describe_image_for_search + embed_text

**What you'll build:** The two data transformation functions that convert a PIL Image into a float vector ready for the index.

**Why it matters:** The quality of search results depends on the richness of the description and the alignment of the embedding space. These two functions are where that quality is determined.

In [ ]:
import base64, hashlib, io
import numpy as np
from PIL import Image

_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def cosine_similarity(a, b):
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)

class ImageIndex:
    def __init__(self):
        self._items = []
    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({'id': image_id, 'description': description,
                            'embedding': np.array(embedding, dtype=np.float32),
                            'metadata': metadata or {}})
    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item) for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [{'id': item['id'], 'description': item['description'],
                 'score': float(score), 'metadata': item['metadata']}
                for score, item in top]
    def __len__(self):
        return len(self._items)


import hashlib
def _mock_describe(img_b64, prompt):
    h = int(hashlib.md5(img_b64.encode()).hexdigest()[:4], 16)
    labels = ['a red apple on a table', 'a blue ocean wave',
              'a green forest path', 'a yellow sunflower field']
    return labels[h % len(labels)]

def _mock_embed(text):
    h = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
    return [((h >> (i * 8)) & 0xff) / 128.0 - 1.0 for i in range(4)]


## Task

**`describe_image_for_search(img, describe_fn=None) -> str`:**
1. `img_b64 = image_to_base64(img)`
2. If `describe_fn is not None`: `return describe_fn(img_b64, _SEARCH_PROMPT)`
3. Else: `ollama.chat(model='llava', messages=[{role, content: _SEARCH_PROMPT, images: [img_b64]}])` → return `resp['message']['content'].strip()`

**`embed_text(text, embed_fn=None) -> list[float]`:**
1. If `embed_fn is not None`: `return embed_fn(text)`
2. Else: `ollama.embeddings(model='nomic-embed-text', prompt=text)` → return `resp['embedding']`

## Your Implementation

In [ ]:
_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)


def describe_image_for_search(img, describe_fn=None) -> str:
    """Generate a text description of an image for search indexing.

    Args:
        img:         PIL Image
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        Text description string
    """
    raise NotImplementedError


def embed_text(text, embed_fn=None) -> list:
    """Embed text to a float vector.

    Args:
        text:     Input string
        embed_fn: callable(text) -> list[float] for testing
    Returns:
        list of float
    """
    raise NotImplementedError


In [ ]:
_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)


def describe_image_for_search(img, describe_fn=None):
    img_b64 = image_to_base64(img)
    if describe_fn is not None:
        return describe_fn(img_b64, _SEARCH_PROMPT)
    import ollama
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': _SEARCH_PROMPT, 'images': [img_b64]}],
    )
    return resp['message']['content'].strip()


def embed_text(text, embed_fn=None):
    if embed_fn is not None:
        return embed_fn(text)
    import ollama
    resp = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return resp['embedding']


## Automated checks

In [ ]:
score, total = 0, 5
try:
    img = Image.new('RGB', (32, 32), 'tomato')

    # describe returns a string
    desc = describe_image_for_search(img, describe_fn=_mock_describe)
    assert isinstance(desc, str) and len(desc) > 0
    score += 1; print("\u2705 describe_image_for_search returns non-empty string")

    # describe_fn receives img_b64 (str) and prompt
    received = {}
    def _capture_desc(b64, prompt):
        received['b64']    = b64
        received['prompt'] = prompt
        return 'captured description'
    describe_image_for_search(img, describe_fn=_capture_desc)
    assert isinstance(received.get('b64'), str) and len(received['b64']) > 10
    assert received.get('prompt') == _SEARCH_PROMPT
    score += 1; print("\u2705 describe_fn receives base64 string and correct prompt")

    # embed_text returns a list of floats
    emb = embed_text('a red car', embed_fn=_mock_embed)
    assert isinstance(emb, list) and len(emb) > 0
    assert all(isinstance(v, float) for v in emb)
    score += 1; print("\u2705 embed_text returns list of floats")

    # embed_fn receives the text string
    received2 = {}
    def _capture_emb(text):
        received2['text'] = text
        return [0.1, 0.2]
    embed_text('hello world', embed_fn=_capture_emb)
    assert received2.get('text') == 'hello world'
    score += 1; print("\u2705 embed_fn receives the input text")

    # same text → same embedding (deterministic mock)
    emb1 = embed_text('a sunny beach', embed_fn=_mock_embed)
    emb2 = embed_text('a sunny beach', embed_fn=_mock_embed)
    assert emb1 == emb2
    score += 1; print("\u2705 embed_text is deterministic for same input")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)


def describe_image_for_search(img, describe_fn=None):
    img_b64 = image_to_base64(img)
    if describe_fn is not None:
        return describe_fn(img_b64, _SEARCH_PROMPT)
    import ollama
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': _SEARCH_PROMPT, 'images': [img_b64]}],
    )
    return resp['message']['content'].strip()


def embed_text(text, embed_fn=None):
    if embed_fn is not None:
        return embed_fn(text)
    import ollama
    resp = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return resp['embedding']
```

**Why `.strip()` on the description?** LLMs sometimes prepend or append whitespace or newlines. Stripping keeps the stored descriptions clean and avoids spurious differences in the embedding space.

</details>